# Building an LLM Client — Step by Step

We start with the simplest possible thing: **one LLM call that gets an answer**. Then we wrap it in a reusable async class, add non-streaming/streaming, and finish with error handling.

It works with OpenAI or any OpenAI-compatible endpoint (OpenRouter, Groq, ...).

## 0. Setup

In [19]:
%pip install openai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import os
from dotenv import load_dotenv

# Option A: put OPENAI_API_KEY (and optionally OPENAI_BASE_URL / OPENAI_MODEL)
#          in a .env file next to this notebook.
load_dotenv()

# Option B: set it directly here (remove this after testing!)
# os.environ['OPENAI_API_KEY'] = 'sk-...'
# os.environ['OPENAI_BASE_URL'] = 'https://openrouter.ai/api/v1'
# os.environ['OPENAI_MODEL'] = 'gpt-4o-mini'

print('api key set:', bool(os.getenv('OPENAI_KEY')))
print('base url   :', os.getenv('OPENAI_BASE_URL', 'https://api.openai.com/v1'))
print('model      :', os.getenv('OPENAI_MODEL', 'gpt-4o-mini'))

api key set: True
base url   : https://api.openai.com/v1
model      : gpt-4o-mini


## 1. Just the LLM call — get the answer

No class, no wrapper. Create the client, send one message, print the answer.

In [21]:
import asyncio
from openai import AsyncOpenAI


async def main():
    client = AsyncOpenAI(
        api_key=os.getenv('OPENAI_KEY'),
        base_url=os.getenv('OPENAI_BASE_URL'),
    )

    messages = [
        {'role': 'user', 'content': 'tell me about the moon in one sentence'}
    ]

    response = await client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'),
        messages=messages,
        stream=False,
    )

    answer = response.choices[0].message.content    #.message.content
    print('Answer:', answer)

    await client.close()


await main()

Answer: The Moon is Earth's only natural satellite, approximately 1/6th the size of Earth, and influences tides, stabilizes the planet's axial tilt, and has been a source of fascination and study for centuries.


## 2. Look at the raw response

Before wrapping things in a class, let's see what the API actually returns: `id`, `model`, `created`, `finish_reason`, `role`, and `usage` (token counts).

In [22]:
import os
import asyncio
from openai import AsyncOpenAI

async def main():
    client = AsyncOpenAI(
        api_key=os.getenv('OPENAI_KEY'),
        base_url=os.getenv('OPENAI_BASE_URL'),
    )

    messages = [
        {'role': 'user', 'content': 'tell me about the moon in one sentence'}
    ]

    response = await client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'),
        messages=messages,
        stream=False,
    )

    message = response.choices[0].message

    # --- the raw response with its inside parameters ---
    print('id           :', response.id)
    print('model        :', response.model)
    print('created      :', response.created)
    print('finish_reason:', response.choices[0].finish_reason)
    print('role         :', message.role)
    if response.usage:
        print('usage        : prompt={} completion={} total={}'.format(
            response.usage.prompt_tokens,
            response.usage.completion_tokens,
            response.usage.total_tokens,
        ))
    # ----------------------------------------------------

    print('\nAnswer:', message.content)

    await client.close()



if __name__ == "__main__":
   await main()

id           : chatcmpl-EI6hCOZxeV6qQDlo4L3D9RxbO7ghZ
model        : gpt-4o-mini-2024-07-18
created      : 1787984350
finish_reason: stop
role         : assistant
usage        : prompt=15 completion=34 total=49

Answer: The Moon is Earth's only natural satellite, characterized by its phases, craters, and influence on tides, and it plays a crucial role in both astronomy and cultural mythology.


### 2.1 Full response object — every parameter

Printing a few fields is nice, but let's dump the **entire** response object as JSON. The OpenAI SDK returns pydantic objects, so `response.model_dump()` gives us everything the API sent back: `id`, `object`, `created`, `model`, `choices` (with the full `message`), `usage` (with token details), etc.


In [5]:
import json

async def main():
    client = AsyncOpenAI(
        api_key=os.getenv('OPENAI_KEY'),
        base_url=os.getenv('OPENAI_BASE_URL'),
    )

    messages = [
        {'role': 'user', 'content': 'tell me about the moon in one sentence'}
    ]

    response = await client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'),
        messages=messages,
        stream=False,
    )

    # dump EVERYTHING the API returned
    print(json.dumps(response.model_dump(), indent=2, default=str))

    await client.close()

await main()


{
  "id": "chatcmpl-EI62PwhZzQ1OICE0MQUR3MJbz6rU6",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "The Moon is Earth's only natural satellite, approximately 1/6th the size of Earth, influencing tides and stabilizing the planet's axial tilt.",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1787981821,
  "model": "gpt-4o-mini-2024-07-18",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": "fp_617e1c99b5",
  "usage": {
    "completion_tokens": 30,
    "prompt_tokens": 15,
    "total_tokens": 45,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0,
      "text_tokens": null
 

### 2.2 Full stream chunk — every parameter

The same dump, but for a **streaming** response. Every chunk is printed as JSON, so you can see the `delta` accumulating and `finish_reason` staying `None` until the last chunk.


In [23]:
import json

async def main():
    client = AsyncOpenAI(
        api_key=os.getenv('OPENAI_KEY'),
        base_url=os.getenv('OPENAI_BASE_URL'),
    )

    messages = [
        {'role': 'user', 'content': 'tell me about the moon in one sentence'}
    ]

    stream = await client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'),
        messages=messages,
        stream=True,
    )

    async for chunk in stream:
        print(json.dumps(chunk.model_dump(), indent=2, default=str))
        print('-' * 40)

    await client.close()

await main()


{
  "id": "chatcmpl-EI6oTEsbZC45oDThjGQZltUs42e68",
  "choices": [
    {
      "delta": {
        "content": "",
        "function_call": null,
        "refusal": null,
        "role": "assistant",
        "tool_calls": null
      },
      "finish_reason": null,
      "index": 0,
      "logprobs": null
    }
  ],
  "created": 1787984801,
  "model": "gpt-4o-mini-2024-07-18",
  "object": "chat.completion.chunk",
  "moderation": null,
  "obfuscation": "Lj1uGo",
  "service_tier": "default",
  "system_fingerprint": "fp_ea9d9536ad",
  "usage": null
}
----------------------------------------
{
  "id": "chatcmpl-EI6oTEsbZC45oDThjGQZltUs42e68",
  "choices": [
    {
      "delta": {
        "content": "The",
        "function_call": null,
        "refusal": null,
        "role": null,
        "tool_calls": null
      },
      "finish_reason": null,
      "index": 0,
      "logprobs": null
    }
  ],
  "created": 1787984801,
  "model": "gpt-4o-mini-2024-07-18",
  "object": "chat.completion.chunk"

## 3. Wrap it in a class (skeleton)

Now we move the call into a reusable `LLMClient` class with a **lazy client** — the `AsyncOpenAI` instance is created only when first needed.

In [7]:
class LLMClient:
    """Step 3: class skeleton with a lazy OpenAI client."""

    def __init__(self, api_key=None, base_url=None, model=None, max_retries=3):
        self._api_key = api_key or os.getenv('OPENAI_KEY') or os.getenv('OPENAI_API_KEY')
        self._base_url = base_url or os.getenv('OPENAI_BASE_URL')
        self._model = model or os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
        self._max_retries = max_retries
        self._client = None

    def get_client(self):
        """Lazily build the underlying AsyncOpenAI client."""
        if self._client is None:
            self._client = AsyncOpenAI(
                api_key=self._api_key,
                base_url=self._base_url,
            )
        return self._client

    async def close(self):
        """Release the underlying HTTP connection."""
        if self._client is not None:
            await self._client.close()
            self._client = None

In [8]:
client = LLMClient()

print('model     :', client._model)
print('base_url  :', client._base_url)
print('has key   :', bool(client._api_key))
print('client    :', type(client.get_client()).__name__)

model     : gpt-4o-mini
base_url  : https://api.openai.com/v1
has key   : True
client    : AsyncOpenAI


## 4. Non-streaming response

`_non_stream_response()` sends one request and gets the whole answer back, printing the raw response parameters.

In [9]:
class LLMClient:
    """Step 4: adds the non-streaming chat completion."""

    def __init__(self, api_key=None, base_url=None, model=None, max_retries=3):
        self._api_key = api_key or os.getenv('OPENAI_KEY') or os.getenv('OPENAI_API_KEY')
        self._base_url = base_url or os.getenv('OPENAI_BASE_URL')
        self._model = model or os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
        self._max_retries = max_retries
        self._client = None

    def get_client(self):
        if self._client is None:
            self._client = AsyncOpenAI(api_key=self._api_key, base_url=self._base_url)
        return self._client

    async def close(self):
        if self._client is not None:
            await self._client.close()
            self._client = None

    async def chat_completion(self, messages, stream=False):
        """Step 4: non-streaming only."""
        client = self.get_client()
        kwargs = {'model': self._model, 'messages': messages, 'stream': stream}
        response = await client.chat.completions.create(**kwargs)
        message = response.choices[0].message

        # --- show the raw response with its inside parameters ---
        print('\n--- raw response ---')
        print('  id           :', response.id)
        print('  model        :', response.model)
        print('  created      :', response.created)
        print('  finish_reason:', response.choices[0].finish_reason)
        print('  role         :', message.role)
        if response.usage:
            print('  usage        : prompt={} completion={} total={}'.format(
                response.usage.prompt_tokens,
                response.usage.completion_tokens,
                response.usage.total_tokens,
            ))
        # --------------------------------------------------------

        return message.content

In [10]:
client = LLMClient()
messages = [{'role': 'user', 'content': 'tell me about the moon in one sentence'}]

async def run_non_stream():
    text = await client.chat_completion(messages, stream=False)
    print('\n[assistant]', text)
    await client.close()

await run_non_stream()


--- raw response ---
  id           : chatcmpl-EI6AGXnQx6dPUUrieO0JBJtMCE7Ps
  model        : gpt-4o-mini-2024-07-18
  created      : 1787982308
  finish_reason: stop
  role         : assistant
  usage        : prompt=15 completion=31 total=46

[assistant] The Moon is Earth's only natural satellite, characterized by its rocky surface, phases that change as it orbits our planet, and significant influence on Earth's tides.


## 5. Streaming response

`_stream_response()` is an **async generator**: each chunk of text is yielded as it arrives. The raw chunk parameters are printed first — note `finish_reason` is `None` on intermediate chunks and only set on the final one.

This step also adds the full `chat_completion()` dispatcher plus retry logic (exponential backoff on rate limits / connection errors).

In [16]:
from openai import AsyncOpenAI, APIConnectionError, APIError, RateLimitError
class LLMClient:
    """Step 5: full client - streaming + non-streaming + retry/backoff."""

    def __init__(self, api_key=None, base_url=None, model=None, max_retries=3):
        self._api_key = api_key or os.getenv('OPENAI_KEY') or os.getenv('OPENAI_API_KEY')
        self._base_url = base_url or os.getenv('OPENAI_BASE_URL')
        self._model = model or os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
        self._max_retries = max_retries
        self._client = None

    def get_client(self):
        if self._client is None:
            self._client = AsyncOpenAI(api_key=self._api_key, base_url=self._base_url)
        return self._client

    async def close(self):
        if self._client is not None:
            await self._client.close()
            self._client = None

    async def chat_completion(self, messages, stream=True):
        """Yield plain text chunks, retrying with exponential backoff."""
        client = self.get_client()
        kwargs = {'model': self._model, 'messages': messages, 'stream': stream}

        for attempt in range(self._max_retries + 1):
            try:
                if stream:
                    async for chunk in self._stream_response(client, kwargs):
                        yield chunk
                else:
                    chunk = await self._non_stream_response(client, kwargs)
                    if chunk:
                        yield chunk
                return

            except RateLimitError:
                if attempt < self._max_retries:
                    await asyncio.sleep(2 ** attempt)  # 1s, 2s, 4s...
                    continue
                yield '[error] rate limit exceeded'
                return

            except APIConnectionError as e:
                if attempt < self._max_retries:
                    await asyncio.sleep(2 ** attempt)
                    continue
                yield f'[error] connection error: {e}'
                return

            except APIError as e:
                yield f'[error] api error: {e}'
                return

    async def _stream_response(self, client, kwargs):
        """Stream text deltas and print raw chunk parameters."""
        response = await client.chat.completions.create(**kwargs)

        async for chunk in response:
            # --- raw stream chunk parameters ---
            print('\n--- raw stream chunk ---')
            print('  id           :', chunk.id)
            print('  model        :', chunk.model)
            print('  created      :', chunk.created)
            print('  finish_reason:', chunk.choices[0].finish_reason)
            if getattr(chunk, 'usage', None):
                print('  usage        : prompt={} completion={} total={}'.format(
                    chunk.usage.prompt_tokens,
                    chunk.usage.completion_tokens,
                    chunk.usage.total_tokens,
                ))
            # -----------------------------------

            if not chunk.choices:
                continue

            delta = chunk.choices[0].delta
            if delta.content:
                yield delta.content

    async def _non_stream_response(self, client, kwargs):
        """Get the full response at once and print raw response parameters."""
        response = await client.chat.completions.create(**kwargs)
        message = response.choices[0].message

        print('\n--- raw response ---')
        print('  id           :', response.id)
        print('  model        :', response.model)
        print('  created      :', response.created)
        print('  finish_reason:', response.choices[0].finish_reason)
        print('  role         :', message.role)
        if response.usage:
            print('  usage        : prompt={} completion={} total={}'.format(
                response.usage.prompt_tokens,
                response.usage.completion_tokens,
                response.usage.total_tokens,
            ))

        return message.content

In [17]:
client = LLMClient()
messages = [{'role': 'user', 'content': 'tell me about the moon in one sentence'}]

async def run_stream():
    async for chunk in client.chat_completion(messages, stream=True):
        print(chunk, end='', flush=True)
    print()
    await client.close()

await run_stream()


--- raw stream chunk ---
  id           : chatcmpl-EI6NFSvMKmA5MNNAMFChVI8oTOiPF
  model        : gpt-4o-mini-2024-07-18
  created      : 1787983113
  finish_reason: None

--- raw stream chunk ---
  id           : chatcmpl-EI6NFSvMKmA5MNNAMFChVI8oTOiPF
  model        : gpt-4o-mini-2024-07-18
  created      : 1787983113
  finish_reason: None
The
--- raw stream chunk ---
  id           : chatcmpl-EI6NFSvMKmA5MNNAMFChVI8oTOiPF
  model        : gpt-4o-mini-2024-07-18
  created      : 1787983113
  finish_reason: None
 Moon
--- raw stream chunk ---
  id           : chatcmpl-EI6NFSvMKmA5MNNAMFChVI8oTOiPF
  model        : gpt-4o-mini-2024-07-18
  created      : 1787983113
  finish_reason: None
 is
--- raw stream chunk ---
  id           : chatcmpl-EI6NFSvMKmA5MNNAMFChVI8oTOiPF
  model        : gpt-4o-mini-2024-07-18
  created      : 1787983113
  finish_reason: None
 Earth's
--- raw stream chunk ---
  id           : chatcmpl-EI6NFSvMKmA5MNNAMFChVI8oTOiPF
  model        : gpt-4o-mini-2024-07-18

## 6. Full demo

A quick interactive demo combining everything. Try both `stream=True` and `stream=False`.

In [18]:
client = LLMClient()

async def ask(question, stream):
    messages = [{'role': 'user', 'content': question}]
    print(f'\n[user] {question}')
    print('[assistant] ', end='', flush=True)
    async for chunk in client.chat_completion(messages, stream=stream):
        print(chunk, end='', flush=True)
    print()

await ask('what is the capital of France?', stream=False)
await ask('and why is it famous?', stream=True)

await client.close()


[user] what is the capital of France?
[assistant] 
--- raw response ---
  id           : chatcmpl-EI6NXpxRBhN1Ign5kPYSghl2IBO0I
  model        : gpt-4o-mini-2024-07-18
  created      : 1787983131
  finish_reason: stop
  role         : assistant
  usage        : prompt=14 completion=7 total=21
The capital of France is Paris.

[user] and why is it famous?
[assistant] 
--- raw stream chunk ---
  id           : chatcmpl-EI6NZRipYvl0VCiJYwhyJ5ydHFEL7
  model        : gpt-4o-mini-2024-07-18
  created      : 1787983133
  finish_reason: None

--- raw stream chunk ---
  id           : chatcmpl-EI6NZRipYvl0VCiJYwhyJ5ydHFEL7
  model        : gpt-4o-mini-2024-07-18
  created      : 1787983133
  finish_reason: None
Could
--- raw stream chunk ---
  id           : chatcmpl-EI6NZRipYvl0VCiJYwhyJ5ydHFEL7
  model        : gpt-4o-mini-2024-07-18
  created      : 1787983133
  finish_reason: None
 you
--- raw stream chunk ---
  id           : chatcmpl-EI6NZRipYvl0VCiJYwhyJ5ydHFEL7
  model        : gpt-4o-

## 7. Ollama — local LLM

Ollama runs models **locally** on your machine. There are two ways to talk to it:

1. **OpenAI-compatible endpoint** — Ollama exposes `/v1`, so we can reuse `LLMClient` unchanged, just pointing `base_url` at `http://localhost:11434/v1` and picking a local model.
2. **Native API** — Ollama's own `/api/chat` endpoint (NDJSON streaming). We implement that below with `httpx`.

First make sure Ollama is running: start the app (or `ollama serve`), then pull a small model with `ollama pull llama3.2`.


In [24]:
import httpx

async def check_ollama():
    async with httpx.AsyncClient(timeout=10.0) as client:
        r = await client.get('http://localhost:11434/api/tags')
        if r.status_code != 200:
            print('Ollama not reachable. Start it with: ollama serve')
            return
        models = [m['name'] for m in r.json().get('models', [])]
        print('Ollama is running. Installed models:', models)

await check_ollama()


Ollama is running. Installed models: ['medgemma-multi:latest', 'MedAIBase/MedGemma1.5:4b']


### 7.1 Reuse LLMClient (OpenAI-compatible)

The exact same class from step 5 — only `base_url` and `model` change.


In [ ]:
# Same LLMClient as before, just pointed at Ollama's OpenAI-compatible API.
# ollama_client = LLMClient(
#     base_url='http://localhost:11434/v1',
#     model="medgemma-multi:latest",
# )

# async def ask_ollama(question, stream=False):
#     messages = [{'role': 'user', 'content': question}]
#     print(f'\n[user] {question}')
#     print('[assistant] ', end='', flush=True)
#     async for chunk in ollama_client.chat_completion(messages, stream=stream):
#         print(chunk, end='', flush=True)
#     print()

# await ask_ollama('tell me about the moon in one sentence')
# await ollama_client.close()



[user] tell me about the moon in one sentence
[assistant] 

C:\Users\KIPLATH NISHA\AppData\Local\Temp\ipykernel_19308\3206531533.py:11: RuntimeWarning: coroutine 'LLMClient.chat_completion' was never awaited
  async for chunk in ollama_client.chat_completion(messages, stream=stream):


TypeError: 'async for' requires an object with __aiter__ method, got coroutine

### 7.2 Native OllamaClient with httpx

Ollama's native endpoint is `/api/chat`. Non-streaming returns one JSON object;
streaming returns **newline-delimited JSON** (one object per line, `done: true` on the last line).


In [14]:
import json
import httpx

class OllamaClient:
    """Native Ollama client using httpx (no openai package needed)."""

    def __init__(self, base_url=None, model=None, timeout=60.0):
        self._base_url = base_url or os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
        self._model = model or os.getenv('OLLAMA_DEFAULT_MODEL', "medgemma-multi:latest")
        self._timeout = timeout
        self._client = None

    def get_client(self):
        if self._client is None:
            self._client = httpx.AsyncClient(base_url=self._base_url, timeout=self._timeout)
        return self._client

    async def close(self):
        if self._client is not None:
            await self._client.aclose()
            self._client = None

    async def chat_completion(self, messages, stream=True):
        client = self.get_client()
        payload = {'model': self._model, 'messages': messages, 'stream': stream}
        if stream:
            async for chunk in self._stream_response(client, payload):
                yield chunk
        else:
            chunk = await self._non_stream_response(client, payload)
            if chunk:
                yield chunk

    async def _non_stream_response(self, client, payload):
        response = await client.post('/api/chat', json=payload)
        response.raise_for_status()
        data = response.json()

        # --- raw response parameters ---
        print('\n--- raw ollama response ---')
        print('  model       :', data.get('model'))
        print('  created_at  :', data.get('created_at'))
        print('  done        :', data.get('done'))
        print('  done_reason :', data.get('done_reason'))
        print('  prompt_eval :', data.get('prompt_eval_count'))
        print('  eval_count  :', data.get('eval_count'))
        print('  eval_duration:', data.get('eval_duration'))
        # --------------------------------

        return data.get('message', {}).get('content')

    async def _stream_response(self, client, payload):
        async with client.stream('POST', '/api/chat', json=payload) as response:
            async for line in response.aiter_lines():
                if not line:
                    continue
                data = json.loads(line)

                # --- raw chunk parameters ---
                if data.get('message') and data['message'].get('content'):
                    content = data['message']['content']
                    print('--- raw ollama chunk ---')
                    print('  model  :', data.get('model'))
                    print('  done   :', data.get('done'))
                    print('  content:', repr(content[:40]))
                    yield content
                elif data.get('done'):
                    print('--- final chunk ---')
                    print('  done_reason :', data.get('done_reason'))
                    print('  eval_count  :', data.get('eval_count'))
                    print('  eval_duration:', data.get('eval_duration'))
                    break


In [26]:
ollama = OllamaClient()
messages = [{'role': 'user', 'content': 'tell me about the moon in one sentence'}]

async def run_ollama_stream():
    async for chunk in ollama.chat_completion(messages, stream=True):
        print(chunk, end='', flush=True)
    print()
    await ollama.close()

await run_ollama_stream()


--- raw ollama chunk ---
  model  : medgemma-multi:latest
  done   : False
  content: 'The'
The--- raw ollama chunk ---
  model  : medgemma-multi:latest
  done   : False
  content: ' Moon'
 Moon--- raw ollama chunk ---
  model  : medgemma-multi:latest
  done   : False
  content: ' is'
 is--- raw ollama chunk ---
  model  : medgemma-multi:latest
  done   : False
  content: ' Earth'
 Earth--- raw ollama chunk ---
  model  : medgemma-multi:latest
  done   : False
  content: "'"
'--- raw ollama chunk ---
  model  : medgemma-multi:latest
  done   : False
  content: 's'
s--- raw ollama chunk ---
  model  : medgemma-multi:latest
  done   : False
  content: ' only'
 only--- raw ollama chunk ---
  model  : medgemma-multi:latest
  done   : False
  content: ' natural'
 natural--- raw ollama chunk ---
  model  : medgemma-multi:latest
  done   : False
  content: ' satellite'
 satellite--- raw ollama chunk ---
  model  : medgemma-multi:latest
  done   : False
  content: ','
,--- raw ollama chunk ---

In [27]:
ollama = OllamaClient()
messages = [{'role': 'user', 'content': 'tell me about the moon in one sentence'}]

async def run_ollama_stream():
    async for chunk in ollama.chat_completion(messages, stream=False):
        print(chunk, end='', flush=True)
    print()
    await ollama.close()

await run_ollama_stream()


--- raw ollama response ---
  model       : medgemma-multi:latest
  created_at  : 2026-08-29T06:45:51.5771457Z
  done        : True
  done_reason : stop
  prompt_eval : 53
  eval_count  : 23
  eval_duration: 704582300
The Moon is Earth's only natural satellite, a celestial body that orbits our planet and influences tides.



## Recap

- **Step 1** — bare LLM call that gets the answer
- **Step 2** — inspect the raw response parameters (`id`, `model`, `usage`, `finish_reason`)
- **Step 3** — class skeleton with lazy client
- **Step 4** — non-streaming response inside the class
- **Step 5** — streaming response + retry/backoff
- **Step 6** — full demo
- **Step 7** — Ollama (reuse `LLMClient` + native `httpx` client)

To use OpenRouter or another provider, set `OPENAI_BASE_URL` and a matching `OPENAI_MODEL`. For Ollama, point `base_url` at `http://localhost:11434/v1` or use `OllamaClient`.
